In [1]:
# Task 1: Cài đặt và nạp các thư viện cần thiết
import os
import time
import hashlib
import sqlite3
import requests
from datetime import datetime
from collections import deque
from urllib.parse import urlparse, urljoin
from bs4 import BeautifulSoup
import re

print("Import thư viện thành công!")

Import thư viện thành công!


In [2]:
TOPIC = "Science"
SEED_URL = "https://plos.org/our-journals/"
ALLOWED_DOMAINS = [
    "plos.org",
    "journals.plos.org"
]
MAX_DEPTH = 3
MAX_PAGES = 300
REQUEST_TIMEOUT = 10
CRAWL_DELAY = 1.0
DB_PATH = "data/plos_crawler.db"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}
print("=== CRAWLER CONFIGURATION ===")
print(f"Topic          : {TOPIC}")
print(f"Seed URL       : {SEED_URL}")
print(f"Allowed Domains: {ALLOWED_DOMAINS}")
print(f"Max Depth      : {MAX_DEPTH} | Max Pages: {MAX_PAGES}")
print(f"Crawl Delay    : {CRAWL_DELAY}s")

=== CRAWLER CONFIGURATION ===
Topic          : Science
Seed URL       : https://plos.org/our-journals/
Allowed Domains: ['plos.org', 'journals.plos.org']
Max Depth      : 3 | Max Pages: 300
Crawl Delay    : 1.0s


In [3]:
# Task 2, 6, 7: URL Frontier sử dụng deque (BFS) và set visited
class URLFrontier:
    def __init__(self, max_depth=3):
        self.queue = deque()
        self.visited = set()
        self.max_depth = max_depth
        self.discovered_count = 0
        self.skipped_count = 0

    def add_url(self, url, depth):
        self.discovered_count += 1
        if depth > self.max_depth or url in self.visited:
            self.skipped_count += 1
            return False
        self.visited.add(url)
        self.queue.append((url, depth))
        return True

    def get_next(self):
        return self.queue.popleft() if self.queue else None

    def is_empty(self):
        return len(self.queue) == 0

frontier = URLFrontier(max_depth=MAX_DEPTH)
print("Đã khởi tạo URL Frontier.")

Đã khởi tạo URL Frontier.


In [4]:
# Duplicate Detection: Kiểm tra exact duplicate nội dung bằng băm SHA-256
class DuplicateDetector:
    def __init__(self):
        self.fingerprints = set()

    def is_duplicate(self, text_content):
        if not text_content:
            return True
        fp = hashlib.sha256(text_content.strip().encode("utf-8")).hexdigest()
        if fp in self.fingerprints:
            return True
        self.fingerprints.add(fp)
        return False

detector = DuplicateDetector()
print("Đã khởi tạo Duplicate Detector.")

Đã khởi tạo Duplicate Detector.


In [5]:
import os
import sqlite3

class Database:
    def __init__(self, db_path):
        # Đảm bảo thư mục tồn tại
        db_dir = os.path.dirname(db_path)
        if db_dir:
            os.makedirs(db_dir, exist_ok=True)

        self.conn = sqlite3.connect(db_path)
        self.create_tables()

    def create_tables(self):
        cursor = self.conn.cursor()

        # Bảng lưu thông tin chi tiết bài báo/trang web
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS pages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                url TEXT UNIQUE,
                domain TEXT,
                title TEXT,
                abstract TEXT,        -- Thêm riêng cho Tóm tắt (Phục vụ Topic Modeling & Summarization)
                authors TEXT,         -- Thêm riêng cho Tác giả
                content TEXT,         -- Lưu Full-text hoặc Cleaned Text
                is_article INTEGER DEFAULT 0, -- Đánh dấu 1 nếu là bài báo khoa học chuẩn, 0 nếu là trang web thường
                depth INTEGER,
                status_code INTEGER,
                crawled_at TEXT
            )
        ''')

        # Bảng lưu quan hệ liên kết giữa các trang (Graph)
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS links (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                source_url TEXT,
                target_url TEXT
            )
        ''')

        # Tạo Index giúp tăng tốc độ truy vấn link và URL
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_pages_url ON pages(url)')
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_links_source ON links(source_url)')

        self.conn.commit()

    def insert_page(self, url, domain, title, abstract, authors, content, is_article, depth, status_code, crawled_at):
        cursor = self.conn.cursor()
        cursor.execute('''
            INSERT OR IGNORE INTO pages (url, domain, title, abstract, authors, content, is_article, depth, status_code, crawled_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (url, domain, title, abstract, authors, content, is_article, depth, status_code, crawled_at))
        self.conn.commit()

    def insert_links_batch(self, link_tuples):
        """Chèn danh sách liên kết theo lô (Batch) để tối ưu tốc độ ghi đĩa"""
        if not link_tuples:
            return
        cursor = self.conn.cursor()
        cursor.executemany('''
            INSERT INTO links (source_url, target_url)
            VALUES (?, ?)
        ''', link_tuples)
        self.conn.commit()

    def close(self):
        self.conn.close()

# Khởi tạo đối tượng DB
db = Database(DB_PATH)
print("Đã kết nối và tối ưu hóa SQLite Database.")

Đã kết nối và tối ưu hóa SQLite Database.


In [6]:
IGNORED_EXTENSIONS = (
    '.jpg', '.jpeg', '.png', '.gif', '.svg', '.webp', '.ico',
    '.css', '.js', '.zip', '.tar', '.gz', '.pdf', '.mp4', '.mp3'
)
ARTICLE_URL_REGEX = (
    r"^https://journals\.plos\.org/"
    r"[^/]+/article\?id=10\.1371/journal\.[a-zA-Z0-9]+"
    r"\.[a-zA-Z0-9\.\-]+$"
)
def is_article_url(url: str) -> bool:
    """Kiểm tra URL có phải article của một journal PLOS hay không."""
    return bool(re.match(ARTICLE_URL_REGEX, url))
def is_allowed_navigation_url(url: str) -> bool:
    """
    Chỉ cho phép navigation cần thiết của PLOS.
    Không cho phép các search/filter có nguy cơ tạo vòng lặp.
    """

    parsed = urlparse(url)
    path = parsed.path.lower()
    query = parsed.query.lower()
    if parsed.netloc.lower() != "journals.plos.org":
        return False
    blocked_params = [
        "filtersubject=",
        "filtersubjects=",
        "filter=",
        "query=",
        "sortkey=",
        "dir=",
    ]

    if any(param in query for param in blocked_params):
        return False
    blacklists = [
        "/s/",
        "/about",
        "/user/",
        "/author/",
        "/static/",
        "lockss-manifest",
    ]
    if any(black in path for black in blacklists):
        return False
    if "/search" in path:
        return False
    if "/browse" in path:
        return False
    if "page=" in query:
        return False

    return True

def extract_and_filter_links(soup, base_url, allowed_domains):
    valid_links = []

    for tag in soup.find_all("a", href=True):

        href = tag.get("href", "").strip()
        if href.startswith(("mailto:", "javascript:", "tel:", "#")):
            continue
        full_url = urljoin(base_url, href)

        parsed = urlparse(full_url)
        if parsed.scheme not in ("http", "https"):
            continue
        if parsed.netloc.lower() != "journals.plos.org":
            continue
        if any(
            parsed.path.lower().endswith(ext)
            for ext in IGNORED_EXTENSIONS
        ):
            continue
        clean_url = (
            f"{parsed.scheme}://"
            f"{parsed.netloc}"
            f"{parsed.path}"
        )
        if parsed.query:
            clean_url += f"?{parsed.query}"
        clean_url = clean_url.rstrip("/")
        if is_article_url(clean_url):
            valid_links.append(clean_url)
            continue
        if is_allowed_navigation_url(clean_url):
            valid_links.append(clean_url)
    return valid_links
def extract_page_data(html_content: str) -> tuple[bool, dict, BeautifulSoup]:
    soup = BeautifulSoup(html_content, "html.parser")
    doi_tag = soup.find(
        "meta",
        {"name": "citation_doi"}
    )
    if not doi_tag or not doi_tag.get("content"):
        title = (
            soup.title.get_text(strip=True)
            if soup.title
            else "Navigation Page"
        )
        soup_clean = BeautifulSoup(
            html_content,
            "html.parser"
        )
        for tag in soup_clean([
            "script",
            "style",
            "noscript",
            "nav",
            "footer",
            "aside"
        ]):
            tag.decompose()
        content = soup_clean.get_text(
            separator=" ",
            strip=True
        )
        return False, {
            "title": title,
            "abstract": "",
            "authors": "",
            "content": content,
            "is_article": 0
        }, soup
    title_tag = soup.find(
        "meta",
        {"name": "citation_title"}
    )

    title = (
        title_tag["content"].strip()
        if title_tag and title_tag.get("content")
        else (
            soup.title.get_text(strip=True)
            if soup.title
            else "No Title"
        )
    )

    abstract_tag = soup.find(
        "meta",
        {"name": "citation_abstract"}
    )

    abstract = (
        abstract_tag["content"].strip()
        if abstract_tag and abstract_tag.get("content")
        else ""
    )

    authors_tags = soup.find_all(
        "meta",
        {"name": "citation_author"}
    )

    authors_list = [
        tag["content"].strip()
        for tag in authors_tags
        if tag.get("content")
    ]
    authors_str = ", ".join(authors_list)
    soup_clean = BeautifulSoup(
        html_content,
        "html.parser"
    )
    for tag in soup_clean([
        "script",
        "style",
        "noscript",
        "nav",
        "footer",
        "aside"
    ]):
        tag.decompose()
    content = soup_clean.get_text(
        separator=" ",
        strip=True
    )
    return True, {
        "title": title,
        "abstract": abstract,
        "authors": authors_str,
        "content": content,
        "is_article": 1
    }, soup
print("Đã cập nhật logic kiểm tra và trích xuất dữ liệu bài báo PLOS.")

Đã cập nhật logic kiểm tra và trích xuất dữ liệu bài báo PLOS.


In [7]:
import time
from datetime import datetime
from urllib.parse import urlparse, parse_qs, urljoin
import requests

# Khởi tạo lại các thống kê
crawled_count = 0
failed_requests = 0
status_code_stats = {}
depth_stats = {d: 0 for d in range(MAX_DEPTH + 1)}

# Quản lý URL:
# - visited_urls: Các URL đã gửi request HTTP GET thành công (dùng cho cả trang NAV và ARTICLE)
# - visited_articles: Các URL bài báo đã được LƯU THÀNH CÔNG vào DB
visited_urls = set()
visited_articles = set()

def canonicalize_url(url: str) -> str:
    parsed = urlparse(url)
    scheme = parsed.scheme.lower()
    netloc = parsed.netloc.lower()
    path = parsed.path.rstrip("/")
    if netloc == "journals.plos.org":
        if "/article" in path:

            params = parse_qs(parsed.query)
            article_id = params.get("id", [None])[0]

            if article_id and article_id.startswith("10.1371/"):
                journal_path = path.split("/article", 1)[0]

                return (
                    f"https://journals.plos.org"
                    f"{journal_path}"
                    f"/article?id={article_id}"
                )
    return f"{scheme}://{netloc}{path}"


# Nạp duy nhất SEED_URL (HTML) vào frontier ban đầu
frontier.add_url(SEED_URL, depth=0)

print("=== BẮT ĐẦU CRAWL PLOS ===")
while not frontier.is_empty() and crawled_count < MAX_PAGES:
    current_url, current_depth = frontier.get_next()

    # 1. CHUẨN HÓA URL
    canonical_current_url = canonicalize_url(current_url)

    # Bỏ qua nếu URL này đã từng request hoặc bài báo này đã được cào thành công
    if canonical_current_url in visited_urls or canonical_current_url in visited_articles:
        continue

    start_time = time.time()

    try:
        response = requests.get(canonical_current_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        elapsed = time.time() - start_time
        status_code = response.status_code
    except requests.RequestException as e:
        failed_requests += 1
        print(f"[Error] {canonical_current_url}: {e}")
        continue

    status_code_stats[status_code] = status_code_stats.get(status_code, 0) + 1

    if "text/html" not in response.headers.get("Content-Type", ""):
        continue

    if status_code == 200:
        # Đánh dấu đã gửi HTTP Request tới URL này
        visited_urls.add(canonical_current_url)
        response.encoding = "utf-8"

        # Bóc tách HTML thu thập thông tin trang
        is_article, article_data, soup = extract_page_data(response.text)
        crawled_at = datetime.now().isoformat()
        domain = urlparse(canonical_current_url).netloc

        # 2. TRÍCH XUẤT LINK CON VÀ NẠP VÀO FRONTIER TRƯỚC
        # (Làm bước này trước khi check duplicate để không ngắt nhánh cây liên kết)
        links = extract_and_filter_links(soup, canonical_current_url, ALLOWED_DOMAINS)
        canonical_links = list(set([canonicalize_url(l) for l in links]))

        # Lưu quan hệ liên kết (Link graph) vào DB
        link_tuples = [(canonical_current_url, link) for link in canonical_links]
        if hasattr(db, 'insert_links_batch'):
            db.insert_links_batch(link_tuples)
        else:
            for link in canonical_links:
                db.insert_link(canonical_current_url, link)

        # Nạp các link con mới thu thập được vào Frontier
        if current_depth < MAX_DEPTH:
            for link in canonical_links:
                if link not in visited_urls and link not in visited_articles:
                    frontier.add_url(link, depth=current_depth + 1)

        # 3. PHÂN LOẠI XỬ LÝ (ARTICLE PAGE VS NAV PAGE)
        if is_article:
            # KIỂM TRA TRÙNG NỘI DUNG DÀNH CHO BÀI BÁO
            if detector.is_duplicate(article_data["content"]):
                print(f"[Duplicate Article Content Skip] {canonical_current_url}")
                continue

            # ĐÁNH DẤU BÀI BÁO ĐÃ XỬ LÝ THÀNH CÔNG (Tránh cào lại qua sub-tab khác)
            visited_articles.add(canonical_current_url)
            crawled_count += 1
            depth_stats[current_depth] = depth_stats.get(current_depth, 0) + 1

            # Lưu vào Database
            db.insert_page(
                url=canonical_current_url,
                domain=domain,
                title=article_data["title"],
                abstract=article_data["abstract"],
                authors=article_data["authors"],
                content=article_data["content"],
                is_article=1,
                depth=current_depth,
                status_code=status_code,
                crawled_at=crawled_at
            )
            print(f"[{crawled_count:03d}] [ARTICLE] Depth: {current_depth} | Links: {len(canonical_links)} | Time: {elapsed:.2f}s | {article_data['title'][:40]}... | URL: {canonical_current_url}")

        else:
            # TRANG ĐIỀU HƯỚNG / SEARCH HTML: Không kiểm tra trùng lặp nội dung
            db.insert_page(
                url=canonical_current_url,
                domain=domain,
                title=article_data["title"],
                abstract="",
                authors="",
                content=article_data["content"],
                is_article=0,
                depth=current_depth,
                status_code=status_code,
                crawled_at=crawled_at
            )
            print(f"[NAV PAGE] Depth: {current_depth} | Links: {len(canonical_links)} | Navigating: {canonical_current_url}")

    else:
        failed_requests += 1

    time.sleep(CRAWL_DELAY)

# Task 10: In báo cáo tổng kết
print("\n" + "=" * 25 + " CRAWLING SUMMARY " + "=" * 25)
print(f"Topic Focus            : {TOPIC}")
print(f"Seed URL               : {SEED_URL}")
print(f"Scientific Articles    : {crawled_count} / {MAX_PAGES} (Target)")
print(f"Unique URLs Discovered : {getattr(frontier, 'discovered_count', len(visited_urls))}")
print(f"Skipped URLs           : {getattr(frontier, 'skipped_count', 0)}")
print(f"Failed Requests        : {failed_requests}")

print("\n--- Distribution by Depth ---")
for d in range(MAX_DEPTH + 1):
    print(f"  - Depth {d}: {depth_stats.get(d, 0)} articles")

print("\n--- HTTP Status Codes ---")
for code, cnt in sorted(status_code_stats.items()):
    print(f"  - HTTP {code}: {cnt}")

print("=" * 68)

=== BẮT ĐẦU CRAWL PLOS ===
[NAV PAGE] Depth: 0 | Links: 16 | Navigating: https://plos.org/our-journals
[NAV PAGE] Depth: 1 | Links: 38 | Navigating: https://journals.plos.org/globalpublichealth
[NAV PAGE] Depth: 1 | Links: 29 | Navigating: https://journals.plos.org/mentalhealth
[NAV PAGE] Depth: 1 | Links: 35 | Navigating: https://journals.plos.org/plosntds
[NAV PAGE] Depth: 1 | Links: 40 | Navigating: https://journals.plos.org/climate
[NAV PAGE] Depth: 1 | Links: 32 | Navigating: https://journals.plos.org/digitalhealth
[NAV PAGE] Depth: 1 | Links: 36 | Navigating: https://journals.plos.org/ploscompbiol
[NAV PAGE] Depth: 1 | Links: 29 | Navigating: https://journals.plos.org/complexsystems
[NAV PAGE] Depth: 1 | Links: 29 | Navigating: https://journals.plos.org/water
[NAV PAGE] Depth: 1 | Links: 26 | Navigating: https://journals.plos.org/ecosystems
[NAV PAGE] Depth: 1 | Links: 36 | Navigating: https://journals.plos.org/plospathogens
[NAV PAGE] Depth: 1 | Links: 32 | Navigating: https://j

In [8]:
# Cell 8: Kiểm tra dữ liệu an toàn (tự động kết nối lại nếu đã đóng)
import sqlite3
import pandas as pd

# 1. Mở lại kết nối nếu db đã bị đóng
try:
    cursor = db.conn.cursor()
except (sqlite3.ProgrammingError, AttributeError):
    print("Kết nối đã bị đóng từ trước, đang kết nối lại...")
    db = Database(DB_PATH)
    cursor = db.conn.cursor()

# 2. Kiểm tra tổng số dòng trong 2 bảng
cursor.execute("SELECT COUNT(*) FROM pages")
total_pages = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM pages WHERE is_article = 1")
total_articles = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM links")
total_links = cursor.fetchone()[0]

print(f"-> Tổng số trang trong DB (tính cả nav): {total_pages}")
print(f"-> Tổng số BÀI BÁO KHOA HỌC chuẩn (is_article = 1): {total_articles}")
print(f"-> Tổng số liên kết trong bảng 'links': {total_links}\n")

# 3. Hiển thị 5 BÀI BÁO KHOA HỌC vừa crawl được (Lọc is_article = 1)
if total_articles > 0:
    query = """
        SELECT id, title, authors, abstract, url, depth, crawled_at
        FROM pages
        WHERE is_article = 1
        LIMIT 5
    """
    df_articles = pd.read_sql_query(query, db.conn)

    # Cắt ngắn abstract hiển thị cho gọn bảng
    df_articles['abstract'] = df_articles['abstract'].apply(lambda x: x[:100] + '...' if len(x) > 100 else x)
    display(df_articles)
else:
    print("⚠️ Cảnh báo: Chưa có BÀI BÁO KHOA HỌC nào được lưu. Hãy kiểm tra lại lượt crawl!")

# 4. Đóng kết nối khi hoàn thành
db.close()
print("Đã đóng kết nối cơ sở dữ liệu an toàn.")

-> Tổng số trang trong DB (tính cả nav): 331
-> Tổng số BÀI BÁO KHOA HỌC chuẩn (is_article = 1): 300
-> Tổng số liên kết trong bảng 'links': 9552



,id,title,authors,abstract,url,depth,crawled_at
0,18,Delays in tuberculosis diagnosis and treatment...,"Mohammad Abdullah Heel Kafi, Poshan Thapa, Mad...",Tuberculosis (TB) care for people with TB in I...,https://journals.plos.org/globalpublichealth/a...,2,2026-10-04T19:51:32.680421
1,19,The public health utility of whole genome sequ...,"Daisy Wang, Angeline Ferdinand, Justin T. Denh...",Whole Genome Sequencing (WGS) is increasingly ...,https://journals.plos.org/globalpublichealth/a...,2,2026-10-04T19:51:35.074491
2,20,Building better citation practices in public h...,"Ella August, Andrea Mahieu, Andrew F. Brouwer","Discovering, selecting, and citing sources to ...",https://journals.plos.org/globalpublichealth/a...,2,2026-10-04T19:51:37.187722
3,21,From Accra to America: Putting development fir...,"Jirair Ratevosian, William Nii Ayitey Menson",,https://journals.plos.org/globalpublichealth/a...,2,2026-10-04T19:51:39.348386
4,22,Physiotherapists are underutilised in health-s...,"María Eugenia de León Pérez, Moisés Jonathan M...",Gender-based violence (GBV) is a major global ...,https://journals.plos.org/globalpublichealth/a...,2,2026-10-04T19:51:42.300413


Đã đóng kết nối cơ sở dữ liệu an toàn.
